# Data Preprocessing for Model Selection
## Telecom Customer Churn Prediction

## Why this notebook exists

EDA told us *what* drives churn: low tenure, month-to-month contracts, missing
security/support add-ons, and electronic-check payments. But a machine
learning model cannot read text like `"Month-to-month"` or work with features
on wildly different scales (`tenure` in single digits, `TotalCharges` in the
thousands). This notebook's job is to turn the cleaned, human-readable dataset
into a numeric, model-ready format -- **without leaking any information from
the future (the test set) into training.**

Every step below answers one question: *"why do I need to do this before I can
train a model?"*

**Input:** `data/processed/cleaned_churn.csv` (output of the cleaning notebook)
**Output:**
- `X_train_scaled.csv`, `X_test_scaled.csv`, `y_train.csv`, `y_test.csv`
- `models/preprocessor.pkl` -- the fitted encoder/scaler, saved so the exact
  same transformation can be reused for model training and later in the
  deployed API (this is what prevents train/serve mismatch in production)

---
## Step 1: Setup

**Why this step:** load the libraries needed to encode, scale and split the
data, and load the cleaned dataset produced by the previous notebook.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = "../data/processed/cleaned_churn.csv"
OUTPUT_DIR = "../data/processed"
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(7032, 20)


   gender SeniorCitizen Partner  ... MonthlyCharges  TotalCharges Churn
0  Female            No     Yes  ...          29.85         29.85    No
1    Male            No      No  ...          56.95       1889.50    No
2    Male            No      No  ...          53.85        108.15   Yes
3    Male            No      No  ...          42.30       1840.75    No
4  Female            No      No  ...          70.70        151.65   Yes

[5 rows x 20 columns]

---
## Step 2: Encode the target variable

**Why this step:** scikit-learn models expect the target to be numeric, not
text. `Churn` is currently `"Yes"` / `"No"`, so I map it to `1` / `0`. Doing
this once here, right after loading, avoids repeating the mapping later in
training or evaluation.

In [2]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

---
## Step 3: Separate numeric and categorical features

**Why this step:** numeric and categorical features need completely
different preprocessing. Numeric features (`tenure`, `MonthlyCharges`,
`TotalCharges`) need **scaling**. Categorical features (`Contract`,
`PaymentMethod`, etc.) need **encoding**. I split them into two lists now so
the next steps can treat each group correctly.

In [3]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in df.columns if c not in numeric_cols + ["Churn"]]

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

Numeric features (3): ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features (16): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


---
## Step 4: Split into train and test -- *before* scaling or encoding anything

**Why this step, and why now specifically:** this is the most important
ordering decision in the whole notebook. If I scaled or encoded the full
dataset first and split afterward, the scaler would learn the mean and
standard deviation using information from the test set too -- a mistake
called **data leakage**. That would make the model look better during
testing than it will actually perform on truly new customers.

By splitting first, the test set stays completely untouched until the very
end, simulating "future, unseen customers" honestly.

I use a **stratified** split so the ~26.6% churn rate found in EDA is
preserved in both the training and test sets -- otherwise, by chance, one
side could end up with a very different churn rate than the other.

In [4]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.3f}")
print(f"Test churn rate:  {y_test.mean():.3f}")

X_train: (5625, 19), X_test: (1407, 19)
Train churn rate: 0.266
Test churn rate:  0.266


---
## Step 5: Build the preprocessing pipeline

**Why this step:** rather than writing separate code to scale numeric columns
and encode categorical columns, I combine both into a single
`ColumnTransformer`. This keeps the transformation as **one reusable object**
that can be saved, reloaded, and applied identically to any new data later
(including live customer data in the deployed API).

- **`StandardScaler`** for numeric features: rescales each column to have
  mean 0 and standard deviation 1. This matters for Logistic Regression,
  which is sensitive to feature magnitude (without it, `TotalCharges` in the
  thousands would dominate over `tenure` in the tens, just because of scale,
  not because it's actually more predictive).
- **`OneHotEncoder`** for categorical features: converts each category into
  its own 0/1 column. `drop="if_binary"` avoids a redundant column for
  features with only two categories (like `Partner`). `handle_unknown="ignore"`
  means that if a brand-new category appears in production (one that wasn't
  in training), the model won't crash -- it just gets encoded as all zeros.

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical_cols),
])
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('cat',
                                 OneHotEncoder(drop='if_binary',
                                               handle_unknown='ignore'),
                                 ['gender', 'SeniorCitizen', 'Partner',
                                  'Dependents', 'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])

---
## Step 6: Fit on the training set only, then transform both sets

**Why this step:** fitting means the scaler/encoder *learns* things like each
numeric column's mean and standard deviation, or which categories exist.
That learning must happen **only on training data**. The test set is only
*transformed* using what was already learned -- it never influences the
transformation itself. This is the practical enforcement of the leakage-safe
split from Step 4.

In [6]:
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_scaled = pd.DataFrame(X_train_arr, columns=feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_arr, columns=feature_names, index=X_test.index)

print(f"Encoded feature count: {X_train_scaled.shape[1]} (started from {X.shape[1]} raw columns)")
X_train_scaled.head()

Encoded feature count: 40 (started from 19 raw columns)


      num__tenure  ...  cat__PaymentMethod_Mailed check
1408     1.321816  ...                              0.0
6992    -0.267410  ...                              0.0
3349     1.444064  ...                              0.0
4486    -1.204646  ...                              0.0
3535     0.669826  ...                              0.0

[5 rows x 40 columns]

---
## Step 7: Validate before saving anything

**Why this step:** before trusting this data enough to train a model on it,
I confirm there are no missing values (one-hot encoding or scaling errors
would show up as NaNs) and that train and test ended up with identical
columns in the identical order -- required for the model to work correctly
on both.

In [7]:
assert X_train_scaled.isnull().sum().sum() == 0
assert X_test_scaled.isnull().sum().sum() == 0
assert list(X_train_scaled.columns) == list(X_test_scaled.columns)
print("Passed: no missing values, and train/test columns match exactly.")

Passed: no missing values, and train/test columns match exactly.


---
## Step 8: Save the processed splits

**Why this step:** saving these files means the next notebook (model
training) doesn't need to repeat any of the steps above -- it just loads
these four files directly and starts training.

In [8]:
X_train_scaled.to_csv(f"{OUTPUT_DIR}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{OUTPUT_DIR}/X_test_scaled.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

for fname in ["X_train_scaled.csv", "X_test_scaled.csv", "y_train.csv", "y_test.csv"]:
    print(fname, "saved:", os.path.exists(f"{OUTPUT_DIR}/{fname}"))

X_train_scaled.csv saved: True
X_test_scaled.csv saved: True
y_train.csv saved: True
y_test.csv saved: True


---
## Step 9: Save the fitted preprocessor -- this is what makes deployment possible

**Why this step:** the CSVs above are only useful for *this* training run.
But in production, new customers arrive as raw, unprocessed data (text
categories, unscaled numbers) -- exactly like the original `cleaned_churn.csv`,
one row at a time. By saving the *fitted* `preprocessor` object itself, the
FastAPI app can later load it and apply the **exact same** scaling and
encoding to a brand-new customer's data before feeding it to the model. This
is what prevents "train/serve skew" -- a common, hard-to-debug bug where a
model behaves differently in production than it did during training because
the preprocessing wasn't applied identically.

In [9]:
preprocessor_path = f"{MODEL_DIR}/preprocessor.pkl"
joblib.dump(preprocessor, preprocessor_path)
print("Saved fitted preprocessor to:", preprocessor_path)

Saved fitted preprocessor to: ../models/preprocessor.pkl


---
## What This Notebook Solves

Starting from a clean but human-readable dataset, this notebook produced a
fully numeric, leakage-safe, model-ready dataset -- and, just as importantly,
a **saved, reusable transformation** that guarantees training and production
will always preprocess data identically.

**Readiness checklist:**
- No missing values
- All categorical features encoded, with safe handling of unseen categories
- Numerical features scaled
- Stratified train/test split, fit only on the training set
- Fitted preprocessor persisted for reuse in training and deployment

**Next notebook:** `03_Model_Training_and_Selection.ipynb` -- train and
compare Logistic Regression, Decision Tree, and Random Forest on this exact
data, and select the best model for deployment.